In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

## Load Trained model 

In [2]:
model = load_model('model.h5')

## load the scaler
with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

## load the encoder
with open('label_encoder_gender.pkl', 'rb') as f:
    encoder = pickle.load(f)

## load the one hot encoder
with open('one_hot_encoder_geography.pkl', 'rb') as f:
    one_hot_encoder = pickle.load(f)

In [3]:
## Example input data
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,  
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [5]:
## One hot encode the Geography column
geography_encoded = one_hot_encoder.transform([[input_data['Geography']]]).toarray()

geography_encoded_df = pd.DataFrame(
    geography_encoded,
    columns=one_hot_encoder.get_feature_names_out(['Geography'])
)

geography_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [6]:
#combine the encoded geography with the rest of the input data
input_df = pd.DataFrame([input_data])
input_df = pd.concat([input_df, geography_encoded_df], axis=1)
input_df.drop('Geography', axis=1, inplace=True)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,Male,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [8]:
## Label encode the Gender column
input_df['Gender'] = encoder.transform(input_df['Gender'])
input_df


,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [10]:
## Scaling the input data
input_scaled = scaler.transform(input_df)
input_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [11]:
## Prediction
prediction = model.predict(input_scaled)
print("Churn Probability:", prediction[0][0])

1/1 [==============================] - 0s 128ms/step
Churn Probability: 0.23223454


In [12]:
if prediction[0][0] > 0.5:
    print("The customer is likely to churn.")
else:
    print("The customer is unlikely to churn.")

The customer is unlikely to churn.
